<p class="lead">Every number a laboratory publishes is an estimate: a quantity computed from the handful of molecules, patients or trajectories that happened to be measured, standing in for a quantity about a population that was not measured. This unit builds the machinery that turns a number into a <em>claim with an interval</em> — the only form in which a number is publishable. We rebuild probability from simulation rather than from formula tables: a distribution is something we sample from and plot, a standard error is something we measure by repeating an experiment thousands of times in silico, and a confidence interval is something whose coverage we check. Three real datasets anchor the whole unit: the ROCK1 bioactivity table the laboratory curates, the replica molecular-dynamics trajectories of the Trp-cage miniprotein, and the scaffold series a medicinal chemist would actually compare. The unit closes with the two facts that most often invalidate an interval in our field — the <strong>experimental noise floor</strong> of a public bioactivity dataset, and the <strong>autocorrelation</strong> of a simulation time series, which makes 500 frames behave like a couple of dozen independent measurements.</p>

<div class="box"><h4>Resumen en español</h4>
<p>Esta unidad reconstruye la probabilidad y la estimación mediante simulación, no mediante tablas de fórmulas. Se repasan variables aleatorias, esperanza y varianza; se reconocen las distribuciones normal, log-normal, binomial, de Poisson y t en datos reales del laboratorio; se demuestran por experimento la ley de los grandes números y el teorema central del límite; se explica por qué las potencias se analizan como pActivity (−log10 de la concentración molar) mostrando que el promedio de los IC50 y el promedio de los pIC50 responden a preguntas distintas; se calculan errores estándar e intervalos de confianza paramétricos y bootstrap (percentil y BCa, con el paquete <code>boot</code>); se introducen resúmenes robustos (mediana, MAD, media recortada) y las correlaciones de Pearson, Spearman y Kendall con sus intervalos. Finalmente se estima el piso de ruido experimental del conjunto ROCK1 y se muestra que en una trayectoria de dinámica molecular 500 fotogramas equivalen a unas pocas decenas de muestras independientes. Esta es la edición en R de la unidad; la edición en Python contiene el mismo material con las mismas figuras.</p></div>

<div class="box key"><h4>Learning objectives</h4>
<ul class="objectives">
<li>Reason with random variables, expectation and variance, and recognise the normal, log-normal, binomial, Poisson and t distributions in laboratory data.</li>
<li>Explain the law of large numbers and the central limit theorem by simulation, and say what each one does and does not promise.</li>
<li>Explain why half-maximal inhibitory concentrations are analysed as pIC50, and show that the mean of IC50 and the mean of pIC50 answer different questions.</li>
<li>Distinguish population, sample, estimator and estimate, and separate sampling variability from measurement error.</li>
<li>Compute and interpret standard errors and confidence intervals analytically and with the percentile and BCa bootstrap, and report uncertainty with every estimate.</li>
<li>Use robust summaries (median, MAD, trimmed mean) and know their breakdown points.</li>
<li>Compute Pearson, Spearman and Kendall correlations with confidence intervals and know which to report.</li>
<li>Estimate the experimental noise floor of a bioactivity dataset and the effective sample size of a correlated time series.</li>
</ul></div>

<div class="box bridge"><h4>How to read this unit</h4>
<p>Read it linearly once, running every chunk and reading the output before the next paragraph; then go back and change one thing at a time — the sample size, the seed, the estimator — and watch the intervals move. Everything runs on base R and its <code>stats</code> package<sup class="c"><a href="#r1">[1]</a></sup>, with <code>data.table</code> for the input/output, <code>ggplot2</code><sup class="c"><a href="#r2">[2]</a></sup> for most figures and the <code>boot</code> package<sup class="c"><a href="#r3">[3]</a></sup> for the resampling intervals of §8. In a fresh Google Colab R runtime run once: <code>install.packages(c("data.table","boot","ggplot2","R.utils"))</code> — in the laboratory's <code>ds-r</code> environment they are already present, and nothing in this unit needs the tidyverse beyond <code>ggplot2</code>. Boxes marked <em>Domain context</em> define the chemistry, biology and simulation vocabulary as it appears; boxes marked <em>Exercise</em> ask you to modify the chunk above them. This unit follows Unit 5 (exploratory analysis) and Unit 6 (curation) and is the foundation of <a href="../U8_Hypothesis_Testing/index.html">Unit 8</a> (hypothesis testing), <a href="../U9_Linear_Models/index.html">Unit 9</a> (linear models) and <a href="../U10_Bayesian/index.html">Unit 10</a> (Bayesian inference).</p></div>



In [ ]:
suppressPackageStartupMessages({
  library(data.table); library(ggplot2); library(boot)
})
set.seed(0)
PAL <- c(blue = "#2b5d9b", orange = "#d2691e", green = "#3a7d44", purple = "#6a4c93")
theme_set(theme_bw(base_size = 9) +
          theme(panel.grid.minor = element_blank(),
                plot.title = element_text(face = "bold", size = 10)))
cat("R", paste(R.version$major, R.version$minor, sep = "."),
    "| data.table", as.character(packageVersion("data.table")),
    "| ggplot2", as.character(packageVersion("ggplot2")),
    "| boot", as.character(packageVersion("boot")), "\n")



The versions are printed so that a reader who reproduces a number three years from now can tell whether a discrepancy comes from the data or from the software. Everything below is deterministic: `set.seed(0)` is called once here, and every chunk that resamples sets its own seed explicitly, because in R a seed is global state and a chunk re-run out of order would otherwise give different numbers.



In [ ]:
D <- function(f) file.path("..", "data", f)
rock   <- fread(D("rock1_chembl_curated.csv"))     ## 1 row per standardised compound
md_obs <- fread(D("md_trpcage_observables.csv"))   ## 6 trajectories x 500 frames
exact  <- rock[censored == FALSE]                  ## compounds with at least one '=' measurement
cat("ROCK1 compounds:", nrow(rock),
    "| with an exact measurement:", nrow(exact),
    "| censored only:", sum(rock$censored), "\n")
cat(sprintf("pActivity (exact subset): mean %.3f, sd %.3f, median %.3f, min %.2f, max %.2f\n",
            mean(exact$pActivity), sd(exact$pActivity), median(exact$pActivity),
            min(exact$pActivity), max(exact$pActivity)))
cat("MD table:", nrow(md_obs), "rows x", ncol(md_obs), "columns | conditions:",
    paste(sort(unique(md_obs$condition)), collapse = ", "), "| replicas:",
    paste(sort(unique(md_obs$replica)), collapse = ", "),
    "| frames per run:", md_obs[, .N, by = .(condition, replica)]$N[1], "\n")



The curated table holds 4,280 compounds, of which 2,764 carry at least one exact (`=`) measurement and 1,516 are censored — reported only as "greater than" or "less than" a bound. The exact subset has a mean pActivity of 6.508 with a standard deviation of 1.109 log units, spanning 3.57 to 9.82, i.e. from about 270 µM to about 150 pM. The molecular-dynamics table is 3,000 rows: six runs (two conditions × three replicas) of 500 saved frames each. Those two tables carry the whole unit.

<div class="box bridge"><h4>Domain context — what a potency measurement is</h4>
<p>A <strong>target</strong> is the protein a drug is meant to act on; ours is <strong>ROCK1</strong> (Rho-associated protein kinase 1), a human enzyme that transfers phosphate groups onto other proteins and whose inhibitors are studied for glaucoma and fibrosis. An <strong>assay</strong> is a laboratory experiment that measures how strongly a molecule interferes with the target. Its usual read-out is the <strong>IC50</strong>: the concentration of compound at which the target's activity is halved, in moles per litre. Smaller is better — a 1 nM compound is a thousand times more potent than a 1 µM one. Closely related read-outs are <strong>Ki</strong> (an inhibition constant, related to IC50 through the Cheng–Prusoff equation<sup class="c"><a href="#r4">[4]</a></sup>) and <strong>Kd</strong> (a binding dissociation constant). Because these numbers span many orders of magnitude, the field converts them to a logarithmic scale: <strong>pActivity = −log<sub>10</sub>(concentration in M)</strong>, so 1 nM = 10<sup>−9</sup> M becomes pActivity 9 and 1 µM becomes 6. Our table, curated from the public ChEMBL database<sup class="c"><a href="#r5">[5]</a></sup>, holds one row per compound with the median pActivity of its measurements. A compound whose measurement is reported only as "IC50 &gt; 10 µM" is <strong>censored</strong>: we know the potency is below a bound but not its value, and such rows are excluded from this unit's estimates (they are flagged by the column <code>censored</code>).</p></div>

## 1. Population, sample, estimator, estimate

Four words do most of the work in this unit, and confusing them is the commonest source of wrong intervals.

A **population** is the set of all values the question is about. It is almost always hypothetical: *the potency against ROCK1 of every molecule our medicinal chemists could synthesise in this series*, *the conformations the Trp-cage miniprotein visits at 300 K*, *the gene-expression profile of inflamed gingival tissue in the human population*. A **sample** is the finite subset we actually measured — 2,764 compounds, 500 frames, 69 patients. An **estimator** is a *rule* for computing something from a sample: "take the arithmetic mean", "take the median", "take the 10 % trimmed mean". An **estimate** is the *number* that rule returns on our particular sample.

The distinction between estimator and estimate is the whole of inferential statistics. The estimate is one number and has no uncertainty — it is what it is. The estimator is a random variable: run the experiment again and the same rule returns a different number. The spread of those hypothetical repetitions is the **sampling distribution**, its standard deviation is the **standard error**, and a confidence interval is a statement about that spread.

Two different sources of variation get folded into the word "uncertainty" and they must be separated:

| source | what it is | how we estimate it here |
|---|---|---|
| **sampling variability** | we measured 2,764 of the possible compounds, not all of them | bootstrap, standard error, CLT |
| **measurement error** | the same compound re-assayed gives a different IC50 | replicate spread within a compound (§11) |

A confidence interval for a mean computed from one assay per compound captures the first and ignores the second. Section 11 measures the second directly, and the comparison is sobering.

The first experiment makes the distinction concrete. We treat the 2,764 measured compounds as if they were the whole population — which they are not, but it lets us *see* the sampling distribution — and draw repeated samples of 25 compounds.



In [ ]:
population <- exact$pActivity
n_draw <- 25; n_rep <- 4000
set.seed(1)
S  <- matrix(sample(population, n_rep * n_draw, replace = TRUE), nrow = n_rep)
mn <- rowMeans(S)
md <- apply(S, 1, median)
dd <- rbind(data.frame(est = mn, which = "sample mean"),
            data.frame(est = md, which = "sample median"))
p <- ggplot(dd, aes(est, fill = which)) +
  geom_histogram(bins = 45, alpha = 0.6, position = "identity") +
  geom_vline(xintercept = mean(population), colour = PAL["blue"], linetype = 2) +
  geom_vline(xintercept = median(population), colour = PAL["orange"], linetype = 2) +
  scale_fill_manual(values = unname(PAL[c("blue", "orange")]), name = NULL) +
  labs(x = "estimate of 'typical' pActivity from n = 25 compounds", y = "count",
       title = "One estimate is a point; the estimator is this whole cloud")
print(p)
cat(sprintf("population mean %.3f, population median %.3f\n", mean(population), median(population)))
cat(sprintf("estimator SD: mean %.4f | median %.4f  (ratio %.2f)\n", sd(mn), sd(md), sd(md) / sd(mn)))
cat(sprintf("sigma/sqrt(n) prediction for the mean: %.4f\n", sd(population) / sqrt(n_draw)))



<p class="caption"><b>Figure 1.</b> Sampling distributions of two estimators of "typical potency", obtained by drawing 4,000 samples of 25 compounds from the 2,764 measured ROCK1 compounds. Both clouds are centred near their respective population values (6.508 for the mean, 6.435 for the median), but they are clouds: a single study of 25 compounds could easily report 6.2 or 6.8. The standard deviation of the mean cloud, 0.2229 log units, is reproduced almost exactly by the formula σ/√n = 0.2217 printed below the figure. The median cloud is 23 % wider (0.2738), which is the price of the robustness bought in §9.</p>

We never see this cloud in a real study, because we get one sample. The rest of the unit reconstructs its width from that single sample — analytically (§7), by resampling (§8), or by knowing the distribution the data came from (§3).

## 2. Random variables, expectation and variance by simulation

A **random variable** is a number produced by a process whose outcome is not determined in advance: the potency of the next compound the chemists make, the number of hydrogen bonds in the next simulation frame, whether a screening hit turns out to be active. Its **expectation** \( \mathbb{E}[X] \) is the long-run average of that number, and its **variance** \( \operatorname{Var}(X)=\mathbb{E}[(X-\mathbb{E}[X])^{2}] \) is the long-run average squared deviation from it; the **standard deviation** \( \sigma=\sqrt{\operatorname{Var}(X)} \) brings it back to the units of \( X \).

Three properties carry nearly all the algebra you need, and all three are worth checking by simulation rather than memorising:

\[ \mathbb{E}[aX+b]=a\,\mathbb{E}[X]+b,\qquad \operatorname{Var}(aX+b)=a^{2}\operatorname{Var}(X),\qquad \operatorname{Var}(X+Y)=\operatorname{Var}(X)+\operatorname{Var}(Y)\ \text{ for independent } X,Y. \]

The third is the one that fails most often in our data, and the failure is the subject of §12: frames of a trajectory are not independent, so the variance of their sum is larger than the sum of their variances, and every uncertainty computed as if they were independent is too small.

The **law of large numbers** (LLN) says that the sample mean of independent draws converges to the expectation as \( n \) grows. It says nothing about how fast, and nothing about the shape of the error — those are the central limit theorem's business (§6).



In [ ]:
p_hit <- 0.08                                   ## an 8 % hit rate in a screening campaign
set.seed(2)
camp <- lapply(1:6, function(k) {
  d <- runif(20000) < p_hit                     ## Bernoulli(0.08): 20,000 compounds tested
  data.frame(n = 1:20000, rate = cumsum(d) / seq_len(20000), campaign = factor(k))
})
camp <- do.call(rbind, camp)
p <- ggplot(camp, aes(n, rate, colour = campaign)) +
  geom_line(linewidth = 0.3, alpha = 0.9) +
  geom_hline(yintercept = p_hit, linetype = 2) +
  scale_x_log10() + coord_cartesian(ylim = c(0, 0.2)) +
  guides(colour = "none") +
  labs(x = "compounds tested (log scale)", y = "running hit rate",
       title = "Law of large numbers: convergence, but slowly")
print(p)
one <- subset(camp, campaign == 1)
for (n in c(100, 1000, 10000, 20000))
  cat(sprintf("after %6d compounds: running hit rate %.4f  (theoretical SE %.4f)\n",
              n, one$rate[n], sqrt(p_hit * (1 - p_hit) / n)))



<p class="caption"><b>Figure 2.</b> Six simulated screening campaigns with a true hit rate of 8 % (dashed line). Each line is the running estimate after the first <i>n</i> compounds. At <i>n</i> = 100 the campaigns disagree by several percentage points — the first one reports 7.0 % — and only after about 10,000 compounds do all six lines settle within roughly one percentage point of the truth (7.82 % at 10,000 and 8.05 % at 20,000 for campaign 1). The printed standard errors show why: the uncertainty falls as 1/√n, from 0.027 at n = 100 to 0.0019 at n = 20,000, so reducing it tenfold costs a hundredfold more experiments.</p>

Note what the law of large numbers does *not* say. It does not say that an unlucky campaign is "due" a hit — the compounds are independent — and it assumes the draws come from the same distribution: if the chemists change the library halfway through, the running average converges to nothing in particular.

## 3. The distributions you will actually meet

Five distributions cover the overwhelming majority of laboratory data. The skill to acquire is recognising which one a column plausibly came from, because that choice determines whether a mean is meaningful, which test applies (Unit 8) and which transform to use.

<div class="box key"><h4>The five, and where they come from</h4>
<ul>
<li><b>Normal</b> — a quantity built by adding many small independent contributions. Physical measurements after transformation, instrument noise, and (by the CLT) averages of almost anything.</li>
<li><b>Log-normal</b> — a quantity built by <i>multiplying</i> many small independent factors, and therefore positive and right-skewed. Concentrations, IC50 values, reaction rates, gene-expression intensities before the log<sub>2</sub>.</li>
<li><b>Binomial</b> — the count of successes in <i>n</i> independent yes/no trials with constant probability. Hits in a screen, actives above a potency threshold, patients responding to a treatment.</li>
<li><b>Poisson</b> — the count of events in a fixed window when events are rare and independent. Hydrogen bonds in a frame, contacts within a cutoff, reads mapped to a gene, mutations per kilobase.</li>
<li><b>Student's <i>t</i></b> — not a model of data but the sampling distribution of a standardised mean when the variance is estimated from the same small sample<sup class="c"><a href="#r6">[6]</a></sup>. It is why small-sample intervals are wider than 1.96 standard errors.</li>
</ul></div>

R names distribution functions systematically, and learning the four prefixes replaces a shelf of tables: `d` for the density, `p` for the cumulative probability, `q` for the quantile (the inverse of `p`) and `r` for random draws, each followed by the distribution's abbreviation — `dnorm`, `pnorm`, `qnorm`, `rnorm`, and likewise `*binom`, `*pois`, `*lnorm`, `*t`. The next chunk exercises all four on the normal distribution, because every interval in this unit is ultimately one `qnorm` or `qt` call.



In [ ]:
mu <- mean(exact$pActivity); sg <- sd(exact$pActivity)
cat(sprintf("density at the mean           dnorm(%.3f) = %.4f\n", mu, dnorm(mu, mu, sg)))
cat(sprintf("P(pActivity > 8) under normal  1 - pnorm(8) = %.4f   (observed %.4f)\n",
            1 - pnorm(8, mu, sg), mean(exact$pActivity > 8)))
cat(sprintf("the 2.5%% and 97.5%% quantiles  qnorm = %.3f, %.3f   (observed %.3f, %.3f)\n",
            qnorm(0.025, mu, sg), qnorm(0.975, mu, sg),
            quantile(exact$pActivity, 0.025), quantile(exact$pActivity, 0.975)))
xs <- seq(2.5, 10.5, length.out = 300)
p <- ggplot(data.frame(x = exact$pActivity), aes(x)) +
  geom_histogram(aes(y = after_stat(density)), bins = 50, fill = "grey75", colour = NA) +
  geom_line(data = data.frame(x = xs, y = dnorm(xs, mu, sg)), aes(x, y),
            colour = PAL["blue"], linewidth = 0.8) +
  geom_vline(xintercept = qnorm(c(0.025, 0.975), mu, sg), colour = PAL["orange"], linetype = 2) +
  labs(x = "pActivity", y = "density",
       title = "d/p/q/r: the fitted normal, its tail probability and its quantiles")
print(p)



<p class="caption"><b>Figure 3.</b> The 2,764 exact pActivity values with a normal density of the same mean (6.508) and standard deviation (1.109), and the fitted 2.5 % and 97.5 % quantiles as dashed lines. The normal model puts 8.91 % of the compounds above pActivity 8 where 9.77 % are observed, and its central 95 % range (4.34–8.68) is close to the empirical one (4.41–8.66) — a good but not perfect fit, with the body of the data slightly more concentrated than the model, which §5 confirms with the negative excess kurtosis.</p>

The next chunk puts four real columns next to the textbook density that claims to describe them.



In [ ]:
ic50_nM <- 10^(9 - exact$pActivity)                 ## back-convert pActivity to nM
hb      <- md_obs[condition == "A", n_hbonds]
rmsd_a1 <- md_obs[condition == "A" & replica == 1, rmsd_ca_nm]
op <- par(mfrow = c(2, 2), mar = c(4, 4, 2.4, 1), cex = 0.75)
h <- hist(log10(ic50_nM), breaks = 50, freq = FALSE, col = "grey80", border = NA,
          main = "IC50 on a log10 scale -> normal", xlab = "log10(IC50 / nM)")
curve(dnorm(x, mean(log10(ic50_nM)), sd(log10(ic50_nM))), add = TRUE, col = PAL["blue"], lwd = 2)
sub <- ic50_nM[ic50_nM < 20000]
hist(sub, breaks = 60, freq = FALSE, col = "grey80", border = NA,
     main = "the same data untransformed -> log-normal", xlab = "IC50 (nM), truncated at 20 uM")
curve(dlnorm(x, mean(log(ic50_nM)), sd(log(ic50_nM))), add = TRUE, col = PAL["blue"], lwd = 2)
tb <- table(hb)
plot(as.integer(names(tb)), as.numeric(tb) / sum(tb), type = "h", lwd = 6, col = "grey80",
     main = "backbone H-bonds per frame vs Poisson", xlab = "n_hbonds (condition A)", ylab = "proportion")
points(as.integer(names(tb)), dpois(as.integer(names(tb)), mean(hb)), col = PAL["orange"], pch = 19, cex = 0.7)
lines(as.integer(names(tb)), dpois(as.integer(names(tb)), mean(hb)), col = PAL["orange"])
hist(rmsd_a1, breaks = 35, freq = FALSE, col = "grey80", border = NA,
     main = "C-alpha RMSD, run A1 vs normal", xlab = "rmsd_ca_nm")
curve(dnorm(x, mean(rmsd_a1), sd(rmsd_a1)), add = TRUE, col = PAL["blue"], lwd = 2)
par(op)
skewness <- function(x) mean((x - mean(x))^3) / mean((x - mean(x))^2)^1.5
cat(sprintf("IC50 (nM): mean %.0f, median %.0f, max %.0f\n", mean(ic50_nM), median(ic50_nM), max(ic50_nM)))
cat(sprintf("log10 IC50: skew %.2f | raw IC50: skew %.2f\n", skewness(log10(ic50_nM)), skewness(ic50_nM)))
cat(sprintf("n_hbonds: mean %.3f, variance %.3f -> index of dispersion %.2f\n",
            mean(hb), var(hb), var(hb) / mean(hb)))



<p class="caption"><b>Figure 4.</b> Four laboratory columns against the textbook density claimed for them. Top left: potencies on the log<sub>10</sub> scale are close to normal (skew −0.06). Top right: the same numbers untransformed are violently right-skewed (skew 7.47, mean 4,176 nM against a median of 367 nM, maximum 270 µM) and the log-normal density follows them. Bottom left: the per-frame count of backbone hydrogen bonds is a count, but the Poisson density with the same mean (8.699) is far too wide — the data are <i>under</i>-dispersed. Bottom right: the C-alpha RMSD of a single 1 ns run is roughly bell-shaped, but that appearance is deceptive, for reasons §12 makes explicit.</p>

The bottom-left panel is the unit's first deliberate failure demonstration. The **index of dispersion**, variance divided by mean, equals 1 for a Poisson variable by construction. The printed value for the hydrogen-bond counts is 0.24, far below 1, meaning the counts cluster much more tightly around their mean than Poisson allows. The reason is physical: a backbone hydrogen bond in a folded miniprotein is not a rare independent event but a structural feature that persists for hundreds of frames, and the number of candidate bonds is bounded by the geometry of a 20-residue chain. "It is a count, therefore it is Poisson" is wrong twice over.



In [ ]:
lam <- mean(hb)
set.seed(3)
sim <- matrix(rpois(2000 * length(hb), lam), nrow = 2000)
sim_disp <- apply(sim, 1, var) / rowMeans(sim)
obs_disp <- var(hb) / mean(hb)
cat(sprintf("observed index of dispersion: %.3f\n", obs_disp))
cat(sprintf("Poisson reference: %.3f  (95%% range %.3f - %.3f over 2000 simulated datasets)\n",
            mean(sim_disp), quantile(sim_disp, 0.025), quantile(sim_disp, 0.975)))
cat(sprintf("fraction of Poisson datasets at least as under-dispersed: %.4f\n", mean(sim_disp <= obs_disp)))
far    <- as.integer(md_obs$dist_W6_P12_nm > 0.70)
blocks <- colSums(matrix(far[seq_len(floor(length(far) / 50) * 50)], nrow = 50))
cat(sprintf("\nrare excursions counted in blocks of 50 frames: mean %.3f, variance %.3f -> dispersion %.2f\n",
            mean(blocks), var(blocks), var(blocks) / mean(blocks)))



The simulation makes the verdict quantitative: not one of 2,000 genuine Poisson datasets of the same size and mean (whose dispersion indices lie between 0.933 and 1.074 in 95 % of cases) is as under-dispersed as the observed hydrogen-bond counts at 0.24, so the Poisson model is rejected without any formal test. The second check shows the opposite failure: counting rare excursions of the cage distance — frames in which the tryptophan–proline distance exceeds 0.70 nm — in blocks of 50 frames gives a dispersion index of 2.88, because the excursions arrive in bursts rather than independently. That is over-dispersion, the usual state of RNA-sequencing counts. One dataset, two opposite violations of the same assumption. Compute the index of dispersion before assuming Poisson; if it is far from 1 in either direction, use the empirical distribution, a negative binomial, or a resampling method.

## 4. Why potencies are analysed as pActivity — a worked example by hand

This is the single most consequential transform in our field, and it is worth tracing on paper before trusting any code. Suppose five laboratories measure the same compound and report IC50 values of **2, 5, 10, 50 and 1,000 nM**. What is "the" potency?

| statistic | computation | value |
|---|---|---|
| arithmetic mean of IC50 | (2 + 5 + 10 + 50 + 1000) / 5 | **213.4 nM** |
| median of IC50 | middle of the sorted list | **10 nM** |
| geometric mean of IC50 | (2 × 5 × 10 × 50 × 1000)<sup>1/5</sup> = 5,000,000<sup>1/5</sup> | **21.87 nM** |
| pIC50 of each value | 9 − log<sub>10</sub>(IC50 in nM) | 8.699, 8.301, 8.000, 7.301, 6.000 |
| mean pIC50 | (8.699 + 8.301 + 8.000 + 7.301 + 6.000) / 5 | **7.660** |
| back-converted mean pIC50 | 10<sup>9 − 7.660</sup> nM | **21.87 nM** |

Three facts fall out of the table, and all three matter.

1. The arithmetic mean of the IC50 values, 213.4 nM, is larger than four of the five measurements. It is not a typical potency; it is dominated by the single weak measurement. In a right-skewed distribution the arithmetic mean is a statement about the *total*, not about the *typical*.
2. The mean of the pIC50 values back-converts exactly to the **geometric mean** of the IC50 values. Averaging on a log scale and taking a geometric mean are the same operation. This is the sense in which "the mean of IC50 and the mean of pIC50 answer different questions": the first asks what the average concentration is, the second asks what the average *order of magnitude* — essentially the average free energy of binding — is.
3. The second question is the one chemistry asks. Binding affinity is related to free energy by \( \Delta G = -RT\ln K \), so a logarithmic potency scale is linear in energy; a ten-fold potency improvement costs roughly the same chemical effort whether it starts at 1 µM or at 10 nM. Errors are also multiplicative — an assay is accurate "to within a factor of two", not "to within 3 nM" — which is exactly the generative mechanism that produces a log-normal distribution<sup class="c"><a href="#r7">[7]</a></sup>.



In [ ]:
vals <- c(2, 5, 10, 50, 1000)                   ## nM, as reported by five laboratories
pv   <- 9 - log10(vals)
gmean <- function(x) exp(mean(log(x)))
cat(sprintf("hand example  arithmetic mean %.2f nM | median %.2f nM | geometric mean %.2f nM\n",
            mean(vals), median(vals), gmean(vals)))
cat("hand example  pIC50 values:", sprintf("%.3f", pv), "\n")
cat(sprintf("hand example  mean pIC50 %.3f -> %.2f nM\n", mean(pv), 10^(9 - mean(pv))))
stopifnot(abs(10^(9 - mean(pv)) - gmean(vals)) < 1e-9)   ## the two are the same number
cat(sprintf("\nwhole ROCK1 dataset (%d compounds with exact measurements):\n", nrow(exact)))
cat(sprintf("  arithmetic mean of IC50-equivalents : %8.1f nM\n", mean(ic50_nM)))
cat(sprintf("  median of IC50-equivalents          : %8.1f nM\n", median(ic50_nM)))
cat(sprintf("  geometric mean of IC50-equivalents  : %8.1f nM\n", gmean(ic50_nM)))
cat(sprintf("  back-converted mean pActivity       : %8.1f nM\n", 10^(9 - mean(exact$pActivity))))
cat(sprintf("  share of the total IC50 sum contributed by the 1%% weakest compounds: %.1f%%\n",
            100 * sum(tail(sort(ic50_nM), floor(length(ic50_nM) / 100))) / sum(ic50_nM)))



The hand-traced example is confirmed to the last digit: the mean pIC50 of 7.660 back-converts to 21.87 nM, exactly the geometric mean, and the `stopifnot` would have stopped the knit if it did not. The pattern then repeats on 2,764 real compounds: the arithmetic mean of the concentrations, 4,175.9 nM, is 13.4 times the geometric mean of 310.8 nM, and the weakest one per cent of the compounds contribute a quarter (25.4 %) of the total concentration sum. Reporting "the mean IC50 of our series is 4.2 µM" would describe a series whose typical member is 367 nM. This is not a subtlety — it is the difference between a publishable summary and a misleading one, and the reason every QSAR target variable in this course is a pActivity.

<div class="box warn"><h4>Common error</h4>
<p>Averaging IC50 values in nanomolar and then taking −log<sub>10</sub> of the average is <em>not</em> the same as averaging pIC50 values, and the difference grows with the spread: here it is the difference between 5.38 and 6.51 pActivity units. The curated table in <code>../data/</code> aggregates replicate measurements with the <em>median of the pActivity values</em>, which is both robust and on the correct scale. When you merge data from two sources yourself, convert to pActivity first, then aggregate.</p></div>

## 5. QQ-plots: deciding whether a transform worked

A histogram tells you whether a distribution looks skewed; a **quantile–quantile plot** tells you *where* and *by how much* it departs from a reference distribution, which is far more actionable. Sort the data, compute for each sorted value the quantile of the reference distribution at the matching cumulative probability, and plot one against the other. If the data follow the reference up to location and scale, the points fall on a straight line; curvature at one end is skew, an S-shape is heavy or light tails. Base R provides `qqnorm()` and `qqline()`, which is all this section needs.



In [ ]:
set.seed(4)
t_sample <- rt(2000, df = 3)
op <- par(mfrow = c(1, 3), mar = c(4, 4, 2.4, 1), cex = 0.75)
for (z in list(list(ic50_nM, "IC50 (nM), untransformed"),
               list(exact$pActivity, "pActivity = 9 - log10(IC50/nM)"),
               list(t_sample, "reference: Student t with 3 df"))) {
  qqnorm(z[[1]], main = z[[2]], pch = 20, cex = 0.3, col = PAL["blue"],
         xlab = "normal theoretical quantile", ylab = "observed")
  qqline(z[[1]], col = PAL["orange"], lwd = 1.6)
}
par(op)
kurtosis <- function(x) mean((x - mean(x))^4) / mean((x - mean(x))^2)^2 - 3
for (z in list(list("IC50 (nM)", ic50_nM), list("pActivity", exact$pActivity)))
  cat(sprintf("%-12s skew %7.2f  excess kurtosis %7.2f  Shapiro-Wilk W %.4f\n",
              z[[1]], skewness(z[[2]]), kurtosis(z[[2]]),
              shapiro.test(z[[2]][1:2000])$statistic))



<p class="caption"><b>Figure 5.</b> Normal QQ-plots. Left: untransformed IC50 values bend sharply upwards at the right — the classic signature of right skew, with the largest observations hundreds of thousands of nanomolar above where a normal distribution would put them. Centre: after the log transform the points lie close to the reference line across more than three orders of magnitude, with mild departures only in the extreme tails. Right: for comparison, a Student <i>t</i> sample with 3 degrees of freedom shows the other characteristic pattern — an S-shape, with both tails heavier than normal. The printed statistics quantify the improvement: skew falls from 7.47 to 0.06, excess kurtosis from 85.5 to −0.44, and the Shapiro–Wilk statistic<sup class="c"><a href="#r8">[8]</a></sup> on the first 2,000 values rises from 0.342 to 0.993. The plot is nevertheless the more useful diagnostic, because it says <i>where</i> the misfit is.</p>

With 2,764 observations a formal normality test rejects almost any real dataset, because real data are never exactly normal and large samples detect tiny departures<sup class="c"><a href="#r9">[9]</a></sup>. Note also that `shapiro.test` in R refuses samples larger than 5,000, which is a deliberate hint from its authors: the useful question is not "are these data normal?" but "is the departure large enough to damage the procedure I am about to run?"

<div class="exercise"><h4>Exercise 1</h4>
<p>Re-run the QQ chunk on the <code>MolWt</code> and <code>TPSA</code> descriptor columns of <code>exact</code> (replace <code>ic50_nM</code> by <code>exact$MolWt</code>). Which of the two is closer to normal, and would a log transform help either of them? Then try <code>QED</code>, a score bounded in [0, 1], and explain why neither a normal nor a log-normal model is appropriate for a bounded quantity (hint: think about what happens near the bounds; the beta distribution of <a href="../U10_Bayesian/index.html">Unit 10</a> is the natural model).</p></div>

## 6. The central limit theorem, and the standard error

The **central limit theorem** (CLT) states that for independent, identically distributed draws with finite variance, the sampling distribution of the mean approaches a normal distribution as \( n \) grows, centred on the population mean, with standard deviation

\[ \operatorname{SE}(\bar{X})=\frac{\sigma}{\sqrt{n}} . \]

Two things are frequently misremembered. The CLT is about the distribution of the *mean*, not about the distribution of the *data* — a sample of 1,000 IC50 values stays log-normal no matter how large \( n \) is. And "approaches as \( n \) grows" hides a rate that depends on the skew of the parent distribution: for a mildly skewed parent \( n = 30 \) is plenty, for the raw IC50 distribution it is nowhere near enough.



In [ ]:
op <- par(mfrow = c(2, 3), mar = c(4, 4, 2.4, 1), cex = 0.72)
set.seed(5)
for (z in list(list(ic50_nM, "raw IC50 (nM)"), list(exact$pActivity, "pActivity"))) {
  dat <- z[[1]]
  for (n in c(2, 10, 50)) {
    m <- rowMeans(matrix(sample(dat, 6000 * n, replace = TRUE), nrow = 6000))
    zz <- (m - mean(dat)) / (sd(dat) / sqrt(n))
    hist(pmin(pmax(zz, -4), 6), breaks = seq(-4, 6, length.out = 61), freq = FALSE,
         col = "grey80", border = NA, xlab = "standardised sample mean", ylab = "density",
         main = sprintf("%s, n = %d (skew %+.2f)", z[[2]], n, skewness(m)))
    curve(dnorm(x), add = TRUE, col = PAL["orange"], lwd = 1.8)
  }
}
par(op)
set.seed(5)
for (z in list(list(ic50_nM, "raw IC50"), list(exact$pActivity, "pActivity"))) {
  sk <- sapply(c(2, 10, 50, 200), function(n)
    skewness(rowMeans(matrix(sample(z[[1]], 6000 * n, replace = TRUE), nrow = 6000))))
  cat(sprintf("%-10s skew of the sample mean at n = 2, 10, 50, 200: %s\n",
              z[[2]], paste(sprintf("%+.2f", sk), collapse = "  ")))
}



<p class="caption"><b>Figure 6.</b> Standardised sampling distributions of the mean (values clipped to the plotting range), with the standard normal density in orange. Top row: for raw IC50 values the sampling distribution is still visibly right-skewed at n = 50 (skew +1.06) and has not vanished even at n = 200 (+0.48), because a single 270 µM compound can dominate the average. Bottom row: on the pActivity scale the skew of the sample mean is already +0.06 at n = 2 and indistinguishable from zero at every larger size. The log transform makes the ordinary machinery of means and intervals <em>valid at the sample sizes we actually have</em>.</p>

This is the practical justification for the transform, independent of the chemical argument in §4. A t-test, a confidence interval and a linear model all lean on the approximate normality of an average; on the raw scale that lean is unsupported for realistic series sizes, and on the log scale it is safe.

## 7. Confidence intervals from a formula, and whether they work

For a mean, the classical 95 % interval is

\[ \bar{x}\pm t_{0.975,\,n-1}\cdot\frac{s}{\sqrt{n}}, \]

where \( s \) is the sample standard deviation and \( t \) the Student quantile — the normal quantile 1.96 inflated to account for having estimated \( \sigma \) from the same small sample<sup class="c"><a href="#r6">[6]</a></sup>. **A 95 % confidence interval is a statement about the procedure, not about the particular interval**: if the whole experiment were repeated many times, 95 % of the intervals constructed this way would contain the true value. The interval in front of you either contains it or does not.



In [ ]:
tab <- data.frame(n = c(3, 5, 10, 25, 100, 1000))
tab$t_quantile <- qt(0.975, tab$n - 1)
tab$inflation_vs_1.96 <- tab$t_quantile / qnorm(0.975)
print(round(tab, 3))
x <- exact$pActivity[1:12]
cat(sprintf("\n12 compounds: mean %.3f, s %.3f, SE %.4f\n", mean(x), sd(x), sd(x) / sqrt(12)))
cat(sprintf("95%% CI by hand  [%.3f, %.3f]\n",
            mean(x) - qt(0.975, 11) * sd(x) / sqrt(12), mean(x) + qt(0.975, 11) * sd(x) / sqrt(12)))
cat(sprintf("95%% CI t.test() [%.3f, %.3f]\n", t.test(x)$conf.int[1], t.test(x)$conf.int[2]))



The table is the whole of Student's correction: with 3 observations the multiplier is 4.303, more than twice the asymptotic 1.96; by n = 25 the inflation is 3 %, and by n = 1,000 it is negligible. The second half of the chunk computes an interval by hand for the first twelve compounds of the table and checks it against `t.test()`, whose `conf.int` component is the same number — useful to know, because `t.test()` is the fastest way to get a parametric interval in R, and the hypothesis test it also prints can simply be ignored.

That promise of 95 % coverage is conditional on assumptions, so it should be *checked* rather than believed. The next chunk checks it by simulation, which is the honest way to find out whether a procedure works on data like yours.



In [ ]:
covers <- function(dat, n, n_rep = 4000, seed = 6) {
  set.seed(seed)
  s    <- matrix(sample(dat, n_rep * n, replace = TRUE), nrow = n_rep)
  m    <- rowMeans(s); sdv <- apply(s, 1, sd)
  half <- qt(0.975, n - 1) * sdv / sqrt(n)
  truth <- mean(dat)
  c(coverage = mean(m - half <= truth & truth <= m + half), width = mean(2 * half))
}
cov <- rbindlist(lapply(c(5, 10, 25, 100), function(n) {
  a <- covers(ic50_nM, n); b <- covers(exact$pActivity, n)
  data.table(n = n, coverage_raw_IC50 = a[["coverage"]], coverage_pActivity = b[["coverage"]],
             mean_width_pActivity = round(b[["width"]], 3))
}))
p <- ggplot(melt(cov[, .(n, raw = coverage_raw_IC50, pActivity = coverage_pActivity)], id.vars = "n"),
            aes(n, value, colour = variable)) +
  geom_line() + geom_point(size = 1.6) +
  geom_hline(yintercept = 0.95, linetype = 2) +
  scale_x_log10() + coord_cartesian(ylim = c(0.5, 1)) +
  scale_colour_manual(values = unname(PAL[c("orange", "blue")]), name = NULL) +
  labs(x = "sample size n", y = "actual coverage of a '95 %' interval",
       title = "A nominal 95 % interval is not always a 95 % interval")
print(p)
print(cov)



<p class="caption"><b>Figure 7.</b> Actual coverage of the nominal 95 % Student interval, measured by resampling from the real data. On the pActivity scale coverage is within half a percentage point of 95 % from n = 5 upwards (0.948, 0.951, 0.953, 0.949). On the raw IC50 scale it is catastrophically low at small n — only 57.1 % of the intervals contain the population mean at n = 5 — and is still only 87.1 % at n = 100. The reason is the skew of Figure 6: for right-skewed data the sample mean and the sample standard deviation are positively correlated, so the samples that land low also get short intervals.</p>

The table under the figure also shows the second thing an interval must be judged on: its **width**. On the pActivity scale the average 95 % interval spans 2.605 log units at n = 5 and 0.438 at n = 100 — a factor of six for a twenty-fold increase in sample size, the 1/√n law again. A procedure can achieve perfect coverage by being uselessly wide, so coverage and width together define a confidence procedure, and the bootstrap of the next section is attractive precisely because it buys better coverage at nearly the same width.

<div class="box key"><h4>What to report</h4>
<p>Report the estimate, the interval and the n: "median pActivity 7.52 (95 % CI 7.41–7.65; n = 207 compounds)". Never report a mean with a standard deviation when the reader wants to know about the mean — the SD describes the spread of the <em>data</em>, the SE describes the spread of the <em>estimate</em>, and they differ by a factor √n<sup class="c"><a href="#r10">[10]</a></sup>. Error bars in figures must state which of the two they are<sup class="c"><a href="#r11">[11]</a></sup><sup class="c"><a href="#r12">[12]</a></sup>, and the convention in the journals this group publishes in is the 95 % CI<sup class="c"><a href="#r13">[13]</a></sup><sup class="c"><a href="#r14">[14]</a></sup>.</p></div>

## 8. The bootstrap

The formula above exists only for a handful of estimators. There is no textbook standard error for a median, for a Spearman correlation, for the area under a ROC curve or for the difference in R² between two models — and those are the quantities we actually report. The **bootstrap**<sup class="c"><a href="#r15">[15]</a></sup><sup class="c"><a href="#r16">[16]</a></sup> solves all of them with one idea: the sample is the best available picture of the population, so resample *from the sample*, with replacement, as many times as the original size, recompute the estimator on each resample, and use the spread of those values as the sampling distribution.

Written out, for \( B \) resamples:

1. draw \( x^{*}_{1},\dots,x^{*}_{n} \) uniformly with replacement from the observed \( x_{1},\dots,x_{n} \);
2. compute \( \hat\theta^{*}_{b} \), the estimator applied to that resample;
3. repeat for \( b = 1,\dots,B \) (use \( B \ge 2{,}000 \) for an interval);
4. the **percentile interval** is the 2.5th and 97.5th percentiles of the \( \hat\theta^{*} \) values.

In R this is the `boot` package<sup class="c"><a href="#r3">[3]</a></sup>, the companion software of Davison and Hinkley's monograph<sup class="c"><a href="#r17">[17]</a></sup>. Its one convention to learn is that the statistic must be a function of the data **and an index vector**, `function(x, i) ...`, because `boot()` passes the resampled row indices rather than the resampled data. We apply it to the question a medicinal chemist actually asks: is the largest scaffold series more potent than the second, and by how much?



In [ ]:
counts <- sort(table(exact$scaffold_smiles), decreasing = TRUE)
s1 <- exact[scaffold_smiles == names(counts)[1], pActivity]
s2 <- exact[scaffold_smiles == names(counts)[2], pActivity]
med <- function(x, i) median(x[i])
set.seed(0); b1 <- boot(s1, med, R = 4000)
set.seed(0); b2 <- boot(s2, med, R = 4000)
ci1 <- boot.ci(b1, type = "perc")$percent[4:5]
ci2 <- boot.ci(b2, type = "perc")$percent[4:5]
dif <- as.vector(b1$t) - as.vector(b2$t)
ci_d <- quantile(dif, c(0.025, 0.975))
reps <- rbind(data.frame(t = as.vector(b1$t), series = sprintf("series 1 (n = %d)", length(s1))),
              data.frame(t = as.vector(b2$t), series = sprintf("series 2 (n = %d)", length(s2))))
p <- ggplot(reps, aes(t, fill = series)) +
  geom_histogram(bins = 45, alpha = 0.75, position = "identity") +
  scale_fill_manual(values = unname(PAL[c("blue", "orange")]), name = NULL) +
  labs(x = "bootstrap replicate of the median pActivity", y = "count",
       title = "Bootstrap sampling distributions of two series medians")
print(p)
cat(sprintf("series 1 median %.3f  95%% CI [%.3f, %.3f]  n = %d\n", median(s1), ci1[1], ci1[2], length(s1)))
cat(sprintf("series 2 median %.3f  95%% CI [%.3f, %.3f]  n = %d\n", median(s2), ci2[1], ci2[2], length(s2)))
cat(sprintf("difference of medians %.3f  95%% CI [%.3f, %.3f]\n",
            median(s1) - median(s2), ci_d[1], ci_d[2]))



<p class="caption"><b>Figure 8.</b> Bootstrap sampling distributions of the median pActivity of the two largest scaffold series, 4,000 resamples each. The clouds barely overlap: series 1 (207 compounds, median 7.521, 95 % CI 7.405–7.648) is 1.294 log units — a factor of twenty in concentration — more potent than series 2 (59 compounds, median 6.227, 95 % CI 5.674–6.398), with a 95 % interval for the difference of 1.072 to 1.851 that excludes zero comfortably. Note the visible granularity of the series-2 distribution: a bootstrap median can only take values that are present in the sample, so with 59 observations the replicate distribution is discrete. That is a real feature of bootstrapping order statistics, not a plotting artefact.</p>

The percentile interval is simple but slightly biased when the estimator's sampling distribution is skewed or when its spread depends on the parameter value. The **bias-corrected and accelerated (BCa)** interval<sup class="c"><a href="#r18">[18]</a></sup><sup class="c"><a href="#r19">[19]</a></sup> corrects both: a bias-correction term \( z_0 \) from the fraction of bootstrap replicates below the observed estimate, and an acceleration term \( a \) from the jackknife skewness. The adjusted percentiles are

\[ \alpha_{1,2}=\Phi\!\left(z_{0}+\frac{z_{0}+z_{\alpha/2,\,1-\alpha/2}}{1-a\,(z_{0}+z_{\alpha/2,\,1-\alpha/2})}\right). \]

It is worth implementing once by hand, both to demystify it and because the implementation is six lines.



In [ ]:
bca_interval <- function(x, stat = median, B = 4000, alpha = 0.05, seed = 0) {
  set.seed(seed); n <- length(x); theta_hat <- stat(x)
  reps <- apply(matrix(sample(x, B * n, replace = TRUE), nrow = B), 1, stat)
  z0   <- qnorm(mean(reps < theta_hat))                      ## bias correction
  jack <- sapply(seq_len(n), function(i) stat(x[-i]))        ## jackknife
  d    <- mean(jack) - jack
  a    <- sum(d^3) / (6 * sum(d^2)^1.5)                      ## acceleration
  zs   <- qnorm(c(alpha / 2, 1 - alpha / 2))
  pct  <- pnorm(z0 + (z0 + zs) / (1 - a * (z0 + zs)))
  list(ci = unname(quantile(reps, pct)), z0 = z0, a = a)
}
hand <- bca_interval(s2)
lib  <- boot.ci(b2, type = "bca")$bca[4:5]
cat(sprintf("series 2 median %.3f\n", median(s2)))
cat(sprintf("  percentile CI   [%.3f, %.3f]\n", ci2[1], ci2[2]))
cat(sprintf("  BCa by hand     [%.3f, %.3f]   (z0 = %+.4f, a = %+.5f)\n",
            hand$ci[1], hand$ci[2], hand$z0, hand$a))
cat(sprintf("  BCa boot.ci     [%.3f, %.3f]\n", lib[1], lib[2]))
cat(sprintf("  Student CI      [%.3f, %.3f]  <- for the MEAN, shown only for scale\n",
            t.test(s2)$conf.int[1], t.test(s2)$conf.int[2]))



The hand-written interval, 5.674 to 6.367, and `boot.ci`'s BCa interval, 5.621 to 6.367, agree to within the Monte-Carlo noise of 4,000 resamples — they are not identical because `boot.ci` applies its adjusted percentiles by interpolating the empirical distribution function rather than by calling `quantile()`, and because the two draw different resamples. Both pull the upper limit down slightly relative to the plain percentile interval (6.398), as the small negative bias correction \( z_0 = -0.164 \) implies. The correction is small here because the median of 59 values is nearly unbiased, but it matters for strongly skewed estimators such as a variance ratio or an enrichment factor. The parametric interval for the *mean* of the same series, 5.808 to 6.331, is shown only to make the point that it answers a different question about a skewed series.

<div class="box warn"><h4>Where the bootstrap fails</h4>
<p>The bootstrap assumes the sample is representative and the estimator is a smooth function of the distribution. It fails for the <strong>maximum</strong> or the minimum (the resampled maximum can never exceed the observed one), degrades for very small <i>n</i>, and is invalid for <strong>dependent data</strong> unless blocks rather than points are resampled — the situation of molecular-dynamics frames (§12). The next chunk exhibits the first failure.</p></div>



In [ ]:
true_max <- max(population)
set.seed(6); small <- sample(population, 40)
set.seed(7)
idx <- matrix(sample(length(small), 4000 * 40, replace = TRUE), nrow = 4000)
reps_max  <- apply(idx, 1, function(i) max(small[i]))
reps_mean <- apply(idx, 1, function(i) mean(small[i]))
op <- par(mfrow = c(1, 2), mar = c(4, 4, 2.4, 1), cex = 0.75)
hist(reps_max, breaks = 30, col = "grey80", border = NA, main = "bootstrap of the MAXIMUM - broken",
     xlab = "bootstrap replicate", ylab = "count")
abline(v = true_max, col = PAL["orange"], lwd = 2, lty = 2)
hist(reps_mean, breaks = 30, col = "grey80", border = NA, main = "bootstrap of the MEAN - fine",
     xlab = "bootstrap replicate", ylab = "count")
abline(v = mean(population), col = PAL["orange"], lwd = 2, lty = 2)
par(op)
cat(sprintf("population maximum %.2f | sample maximum %.2f | bootstrap 95%% CI [%.2f, %.2f]\n",
            true_max, max(small), quantile(reps_max, 0.025), quantile(reps_max, 0.975)))
cat(sprintf("distinct values taken by the bootstrapped maximum: %d\n", length(unique(reps_max))))
cat(sprintf("population mean    %.3f | sample mean    %.3f | bootstrap 95%% CI [%.3f, %.3f]\n",
            mean(population), mean(small), quantile(reps_mean, 0.025), quantile(reps_mean, 0.975)))



<p class="caption"><b>Figure 9.</b> Two bootstraps on the same sample of 40 compounds (dashed line: the population value). Left: the bootstrap distribution of the maximum takes only 7 distinct values, all of them at or below the sample maximum of 8.82, and the resulting interval 7.60–8.82 excludes the population maximum of 9.82 entirely — the estimator depends on the extreme tail of the distribution, which resampling cannot reconstruct. Right: the bootstrap of the mean behaves exactly as advertised, its interval 6.038–6.698 covering the population mean of 6.508. The lesson generalises: before bootstrapping an unfamiliar statistic, ask whether it depends on the bulk of the distribution (safe) or on its edge (unsafe).</p>

<div class="exercise"><h4>Exercise 2</h4>
<p>Write <code>boot_ci(x, stat, B)</code> around <code>boot()</code> and <code>boot.ci()</code> and use it on the <b>interquartile range</b> (<code>IQR</code>) and on the <b>90th percentile</b> (<code>function(x, i) quantile(x[i], 0.9)</code>) of <code>s1</code>. Which of the two gives a smoother bootstrap distribution, and why? Then reduce <code>s1</code> to its first 12 compounds and repeat: at what sample size does the 90th-percentile interval stop being trustworthy? Compare <code>boot.ci</code>'s five interval types on the same object and note which one it refuses to compute and why.</p></div>

## 9. Robust summaries

A single mistyped value, a compound recorded in µM instead of nM, or one genuinely outlying patient can move an arithmetic mean arbitrarily far. The **breakdown point** of an estimator is the fraction of the sample that must be corrupted before the estimate can be driven to infinity: it is 0 for the mean (one bad value is enough), 0.5 for the median, and \( \gamma \) for a \( \gamma \)-trimmed mean. Robust alternatives<sup class="c"><a href="#r20">[20]</a></sup> cost a little efficiency when the data are clean and save the analysis when they are not:

- **median** — the 50th percentile; breakdown 0.5.
- **MAD** — median absolute deviation, \( \operatorname{median}(|x_i - \operatorname{median}(x)|) \), multiplied by 1.4826 so that it estimates σ for normal data<sup class="c"><a href="#r21">[21]</a></sup>. R's `mad()` applies that constant by default, which is worth knowing before comparing it with a hand-written version.
- **trimmed mean** — discard the lowest and highest γ of the data and average the rest; in R, `mean(x, trim = 0.1)`.



In [ ]:
clean <- exact$pActivity
tmean <- function(v) mean(v, trim = 0.1)
set.seed(8)
cont <- clean
bad  <- sample(length(cont), floor(0.01 * length(cont)))
cont[bad] <- cont[bad] - 3                        ## a 1000-fold unit mistake in 1 % of the rows
tab <- data.frame(statistic = c("mean", "median", "10% trimmed mean", "SD", "MAD (x1.4826)"),
                  clean     = c(mean(clean), median(clean), tmean(clean), sd(clean), mad(clean)),
                  corrupted = c(mean(cont),  median(cont),  tmean(cont),  sd(cont),  mad(cont)))
tab$shift <- tab$corrupted - tab$clean
fracs <- seq(0, 0.15, by = 0.015)
set.seed(9)
curve_dt <- rbindlist(lapply(fracs, function(fr) {
  v <- clean
  if (fr > 0) { ii <- sample(length(v), floor(fr * length(v))); v[ii] <- v[ii] - 3 }
  data.table(frac = fr, mean = mean(v), median = median(v), trimmed = tmean(v))
}))
p <- ggplot(melt(curve_dt, id.vars = "frac"), aes(100 * frac, value, colour = variable)) +
  geom_line() + geom_point(size = 1.5) +
  scale_colour_manual(values = unname(PAL[c("orange", "blue", "green")]), name = NULL) +
  labs(x = "% of compounds with a 1000-fold unit error", y = "estimate of typical pActivity",
       title = "Breakdown in practice")
print(p)
knitr::kable(tab, digits = 3)



<p class="caption"><b>Figure 10.</b> Each summary of typical potency as the fraction of corrupted rows grows from 0 to 15 %. The mean (orange) slides down immediately and linearly, from 6.508 to 6.058 — exactly the 0.03 log units per percentage point that a −3 shift in 1 % of the rows must produce. The 10 % trimmed mean (green) holds up better but still loses 0.34 units by 15 %, because the contamination lands inside the data rather than beyond the trimming point. The median (blue) falls by only 0.175 units over the whole range and is completely flat beyond 12 %. The table under the figure shows the same effect for the scale estimates at 1 % contamination: the standard deviation inflates by 0.050 while the MAD-based estimate moves by 0.023.</p>

This is not an argument for always reporting medians: the mean is efficient on clean symmetric data and is what every linear model estimates. The argument is for knowing which you reported and why, and for checking that the two agree — a large gap between mean and median is a diagnostic that the column needs looking at. Here, on clean data, the mean is 6.508 and the median 6.435, a difference of 0.07 log units that reflects the slight asymmetry visible in Figure 3.

## 10. Correlation, and three ways to measure it

Correlation is the most over-interpreted statistic in applied science, so it pays to be precise about what each coefficient measures.

- **Pearson's \( r \)** measures the strength of the *linear* relationship; it is sensitive to outliers and to monotone non-linearity.
- **Spearman's \( \rho \)** is Pearson's \( r \) on the *ranks*: monotone association of any shape, insensitive to outliers in value.
- **Kendall's \( \tau \)** counts concordant minus discordant pairs; it is the most interpretable (a rescaled probability that two compounds are ordered the same way by both variables) and the most stable in small samples.

All three are `cor(x, y, method = ...)` in R, and all three have a test in `cor.test()`. For a confidence interval, Pearson's \( r \) has the Fisher \( z \) transform, \( z=\operatorname{artanh}(r) \), which is approximately normal with standard error \( 1/\sqrt{n-3} \); for the rank coefficients the bootstrap is the practical route<sup class="c"><a href="#r22">[22]</a></sup><sup class="c"><a href="#r23">[23]</a></sup>.



In [ ]:
corr_row <- function(x, y, label, B = 3000, seed = 0) {
  n <- length(x)
  r <- cor(x, y); rho <- cor(x, y, method = "spearman"); tau <- cor(x, y, method = "kendall")
  z <- atanh(r); se <- 1 / sqrt(n - 3)
  ci_r <- tanh(c(z - 1.96 * se, z + 1.96 * se))
  set.seed(seed)
  idx <- matrix(sample(n, B * n, replace = TRUE), nrow = B)
  ci_rho <- quantile(apply(idx, 1, function(i) cor(x[i], y[i], method = "spearman")), c(0.025, 0.975))
  data.table(set = label, n = n, pearson_r = r, r_lo = ci_r[1], r_hi = ci_r[2],
             p_pearson = cor.test(x, y)$p.value, spearman = rho,
             rho_lo = ci_rho[1], rho_hi = ci_rho[2], kendall = tau)
}
sub <- exact[scaffold_smiles == names(counts)[1]]
ctab <- rbind(corr_row(exact$MolLogP, exact$pActivity, "all exact compounds"),
              corr_row(sub$MolLogP, sub$pActivity, "largest series only"))
op <- par(mfrow = c(1, 2), mar = c(4, 4, 2.4, 1), cex = 0.75)
plot(exact$MolLogP, exact$pActivity, pch = 20, cex = 0.25, col = "#55555580",
     xlab = "MolLogP (calculated lipophilicity)", ylab = "pActivity",
     main = sprintf("all %d compounds: r = %.3f", nrow(exact), ctab$pearson_r[1]))
plot(sub$MolLogP, sub$pActivity, pch = 20, cex = 0.7, col = PAL["blue"],
     xlab = "MolLogP (calculated lipophilicity)", ylab = "pActivity",
     main = sprintf("largest series (n = %d): r = %.3f", nrow(sub), ctab$pearson_r[2]))
par(op)
cat(sprintf("Spearman test on all compounds: rho = %.3f, p = %.4g\n",
            ctab$spearman[1],
            suppressWarnings(cor.test(exact$MolLogP, exact$pActivity, method = "spearman"))$p.value))
knitr::kable(ctab, digits = 4)



<p class="caption"><b>Figure 11.</b> Calculated lipophilicity against potency, for all 2,764 measured compounds (left) and for the single largest congeneric series (right). The global Pearson correlation is −0.025 with a 95 % interval from −0.063 to +0.012 that straddles zero (p = 0.18), while the Spearman coefficient, −0.055 with an interval from −0.091 to −0.017, is "statistically significant" at p = 0.004 purely because n is large. Inside the series the coefficients are −0.113 and −0.104 with intervals that include zero (n = 207). The honest reading is: <strong>lipophilicity carries no useful univariate information about ROCK1 potency in this dataset</strong>, and the significant global Spearman value is an illustration of the gap between significance and importance that <a href="../U8_Hypothesis_Testing/index.html">Unit 8</a> develops.</p>

Two habits follow. Always report the interval next to the coefficient: "ρ = −0.055 (95 % CI −0.091 to −0.017, n = 2,764)" makes the irrelevance visible in a way that "ρ = −0.055, p = 0.004" does not. And always plot the data. Anscombe's quartet<sup class="c"><a href="#r24">[24]</a></sup> and its modern successors<sup class="c"><a href="#r25">[25]</a></sup> are datasets with identical means, variances and correlations and utterly different shapes; R ships the quartet as `datasets::anscombe`, so the demonstration costs one chunk.



In [ ]:
a <- datasets::anscombe
op <- par(mfrow = c(1, 4), mar = c(4, 4, 2.4, 1), cex = 0.7)
summ <- rbindlist(lapply(1:4, function(k) {
  x <- a[[paste0("x", k)]]; y <- a[[paste0("y", k)]]
  fit <- lm(y ~ x)
  plot(x, y, pch = 19, cex = 0.9, col = PAL["blue"], xlim = c(3, 20), ylim = c(3, 13),
       main = sprintf("set %d", k), xlab = "x", ylab = if (k == 1) "y" else "")
  abline(fit, col = PAL["orange"], lwd = 1.4)
  data.table(set = k, mean_x = mean(x), mean_y = mean(y), sd_y = sd(y),
             pearson = cor(x, y), spearman = cor(x, y, method = "spearman"),
             intercept = coef(fit)[1], slope = coef(fit)[2])
}))
par(op)
knitr::kable(summ, digits = 3)



<p class="caption"><b>Figure 12.</b> Anscombe's four datasets. All four share a mean x of 9, a mean y of 7.50, a standard deviation of y of about 2.03, a Pearson correlation of 0.816 and the same fitted line y = 3.00 + 0.50x (printed in the table), yet only set 1 is a plausible linear relationship: set 2 is a parabola, set 3 is a perfect line with one outlier, and set 4 has no x variation at all except for one leverage point that determines the entire fit. Note that the Spearman coefficients <em>do</em> differ between the sets — 0.818, 0.691, 0.991 and 0.500 — so rank correlation is not fooled in the same way, which is a small argument for reporting both.</p>

## 11. The experimental noise floor of a bioactivity dataset

A model cannot predict a measurement more precisely than the measurement can reproduce itself. Before any modelling in Units 11–17 it is worth knowing what that limit is for our data, and the curated table lets us estimate it directly: several hundred compounds were measured more than once, often in different laboratories and different publications, and the column `pActivity_sd` records the spread of those repeats.



In [ ]:
rep2   <- rock[n_exact >= 2 & censored == FALSE]
spread <- rep2[pActivity_range > 0]
pooled_sd <- sqrt(mean(rep2$pActivity_sd^2, na.rm = TRUE))
total_sd  <- sd(exact$pActivity)
cat(sprintf("compounds with >= 2 exact measurements : %d\n", nrow(rep2)))
cat(sprintf("  of these, with non-zero spread       : %d  (the rest report identical values)\n", nrow(spread)))
cat(sprintf("  median within-compound SD (spread>0) : %.3f log units\n", median(spread$pActivity_sd)))
cat(sprintf("  pooled within-compound SD            : %.3f log units\n", pooled_sd))
cat(sprintf("  median within-compound range         : %.2f log units\n", median(spread$pActivity_range)))
cat(sprintf("  compounds flagged wide_spread (>1 log): %d\n", sum(rock$wide_spread)))
cat(sprintf("  largest observed range               : %.2f log units\n", max(rep2$pActivity_range)))
cat(sprintf("\nspread of the target itself (SD)       : %.3f log units\n", total_sd))
cat(sprintf("implied ceiling on R^2 for ANY model   : %.3f\n", 1 - pooled_sd^2 / total_sd^2))
cat(sprintf("implied floor on RMSE                  : %.3f log units\n", pooled_sd))
p <- ggplot(spread, aes(pActivity_range)) +
  geom_histogram(bins = 40, fill = "grey75", colour = NA) +
  geom_vline(xintercept = 1, colour = PAL["orange"], linetype = 2) +
  labs(x = "range of repeated pActivity measurements of the same compound (log units)",
       y = "compounds", title = "Two laboratories, one compound, two answers")
print(p)



<p class="caption"><b>Figure 13.</b> Distribution of the within-compound range for the 161 compounds measured more than once with a non-zero spread; the dashed line is the one-log-unit flag that the curation protocol sets. The median range is 0.48 log units — a factor of three in concentration — and a long tail reaches 3.30 log units, i.e. a two-thousand-fold disagreement between two published values for the same molecule. Sixty compounds of the full table exceed the flag.</p>

The two numbers printed above the figure are the ones to remember when a reviewer asks why a QSAR model "only" reaches an R² of 0.6. With a pooled within-compound standard deviation of 0.406 log units against a total target spread of 1.109, a perfect model that predicted the *true* potency of every compound exactly would still score an R² of about 0.866 against these labels, and could not achieve an RMSE below about 0.41 log units. Independent analyses of public ChEMBL data reach the same order of magnitude — around 0.5 log units for mixed public IC50 data<sup class="c"><a href="#r26">[26]</a></sup> and about 0.44 for heterogeneous Ki data<sup class="c"><a href="#r27">[27]</a></sup>.

Three honest caveats belong with that estimate, and stating them is part of the craft:

1. Only 337 compounds have two or more exact measurements, and 176 of those — more than half — report *identical* values, often the same primary measurement propagated through several papers rather than an independent repeat. Including them pulls the pooled estimate down; the median over compounds with genuinely different values, 0.302 log units, is the alternative, and the truth lies between them.
2. Re-measured compounds are not a random sample: they are reference inhibitors, clinical compounds and series anchors, so the estimate may not transfer to a singleton in the tail of the table.
3. The tabulated target is the *median* of the available measurements, so its error is smaller than a single assay's by roughly √n for well-measured compounds.

<div class="box key"><h4>The rule</h4>
<p>Estimate the noise floor of your labels before you tune a model, quote it in the paper, and compare every performance claim to it. A model whose RMSE equals the assay's reproducibility is as good as the data allow; a model whose cross-validated RMSE is <em>below</em> the noise floor is evidence of leakage, not of genius.</p></div>

## 12. When n is not n: correlated samples

Everything above assumed independent draws. Molecular-dynamics frames are not independent: the structure at 402 ps is nearly the structure at 400 ps. The consequence is severe and widely ignored — a 1 ns trajectory saved every 2 ps looks like 500 measurements, so the naive standard error σ/√500 is tiny, and the resulting interval can be an order of magnitude too narrow.

<div class="box bridge"><h4>Domain context — what a molecular-dynamics trajectory is</h4>
<p>A <strong>molecular-dynamics (MD) simulation</strong> integrates Newton's equations for every atom of a protein surrounded by explicit water, in steps of about 2 femtoseconds, and writes the coordinates to disk every few picoseconds. Our frozen dataset holds six 1 ns simulations of the <strong>Trp-cage</strong> miniprotein (PDB 1L2Y, 20 residues)<sup class="c"><a href="#r28">[28]</a></sup> run with OpenMM<sup class="c"><a href="#r29">[29]</a></sup> and the AMBER ff14SB force field<sup class="c"><a href="#r30">[30]</a></sup>: three replicas at 300 K (condition A) and three at 340 K (condition B), differing only in their random seed. The <strong>C-alpha RMSD</strong> is the root-mean-square deviation of the backbone from the starting structure after optimal superposition, in nanometres — the standard scalar summary of "how far the protein has moved". A <strong>replica</strong> is an independent repetition of the same simulation with a different random seed; replicas, not frames, are the independent experimental units, which is the whole point of this section and the centrepiece of <a href="../U8_Hypothesis_Testing/index.html">Unit 8</a>.</p></div>

The tool is the **autocorrelation function** \( C(\Delta) \) of the time series and its integral, the **integrated autocorrelation time** \( \tau_{\text{int}} \). The effective number of independent samples in a run of length \( T \) is approximately \( n_{\text{eff}} = T / (2\tau_{\text{int}}) \), and the correct standard error is \( \sigma/\sqrt{n_{\text{eff}}} \), not \( \sigma/\sqrt{n} \)<sup class="c"><a href="#r31">[31]</a></sup><sup class="c"><a href="#r32">[32]</a></sup>. The same autocorrelation machinery also decides how much of the beginning of a run to discard as equilibration, for which automated criteria exist<sup class="c"><a href="#r33">[33]</a></sup>; Unit 20 applies them.



In [ ]:
tau_int <- function(x) {
  cc <- as.vector(acf(x, lag.max = floor(length(x) / 4), plot = FALSE, demean = TRUE)$acf)
  first_neg <- which(cc < 0)[1]
  if (is.na(first_neg)) first_neg <- length(cc)
  0.5 + sum(cc[2:(first_neg - 1)])              ## in units of the saving interval
}
block_se <- function(x, n_blocks = 10) {
  b <- colMeans(matrix(x[seq_len(floor(length(x) / n_blocks) * n_blocks)], ncol = n_blocks))
  sd(b) / sqrt(n_blocks)
}
run <- md_obs[condition == "A" & replica == 2, rmsd_ca_nm]
dt  <- 2.0                                       ## ps between saved frames
tau <- tau_int(run) * dt
ac  <- acf(run, lag.max = 120, plot = FALSE)
p <- ggplot(data.frame(lag = ac$lag * dt, acf = as.vector(ac$acf)), aes(lag, acf)) +
  geom_line(colour = PAL["blue"]) +
  geom_hline(yintercept = 0) + geom_hline(yintercept = 1 / exp(1), colour = PAL["orange"], linetype = 2) +
  labs(x = "lag (ps)", y = "autocorrelation of C-alpha RMSD",
       title = "Consecutive frames are not independent measurements")
print(p)
cat(sprintf("run A2: %d frames, mean RMSD %.4f nm, SD %.4f nm\n", length(run), mean(run), sd(run)))
cat(sprintf("integrated autocorrelation time tau_int = %.0f ps -> n_eff = %.1f independent samples\n",
            tau, 1000 / (2 * tau)))
cat(sprintf("  naive SE  sigma/sqrt(500)      : %.5f nm\n", sd(run) / sqrt(length(run))))
cat(sprintf("  corrected SE sigma/sqrt(n_eff) : %.5f nm\n", sd(run) / sqrt(1000 / (2 * tau))))
cat(sprintf("  10-block SE                    : %.5f nm\n", block_se(run)))



<p class="caption"><b>Figure 14.</b> Autocorrelation of the C-alpha RMSD of run A2 against lag, with the 1/e line dashed. The correlation does not fall below 1/e until a lag of roughly twenty picoseconds and oscillates around zero thereafter, which means that frames saved 2 ps apart carry almost the same information. The integrated autocorrelation time of this run, estimated by summing the autocorrelation function to its first zero crossing, is 18 ps, so a 1 ns trajectory contains about 28 effectively independent samples, not five hundred. The three standard errors printed below the figure differ by a factor of four — 0.00049 nm naive, 0.00206 nm autocorrelation-corrected and 0.00173 nm from ten blocks — and the naive one is the smallest, which is why it is the one most often reported.</p>

The `boot` package handles dependent data through `tsboot()`, which resamples *blocks* of consecutive observations instead of single points, so that the within-block correlation survives the resampling. The block length must exceed the correlation time; the chunk below shows how much difference that one argument makes.



In [ ]:
set.seed(10)
ib <- boot(run, function(x, i) mean(x[i]), R = 2000)                  ## i.i.d. bootstrap - wrong here
set.seed(10)
bb <- tsboot(run, mean, R = 2000, l = 25, sim = "fixed")               ## 25 frames = 50 ps per block
cat(sprintf("i.i.d. bootstrap SE %.5f nm | 95%% CI [%.4f, %.4f]\n",
            sd(ib$t), quantile(ib$t, 0.025), quantile(ib$t, 0.975)))
cat(sprintf("block bootstrap  SE %.5f nm | 95%% CI [%.4f, %.4f]  (l = 25 frames)\n",
            sd(bb$t), quantile(bb$t, 0.025), quantile(bb$t, 0.975)))
for (l in c(5, 25, 50, 100)) {
  set.seed(10)
  z <- tsboot(run, mean, R = 1000, l = l, sim = "fixed")
  cat(sprintf("  block length %3d frames (%3.0f ps): SE %.5f nm\n", l, l * dt, sd(z$t)))
}



The i.i.d. bootstrap reproduces the naive standard error almost exactly (0.00050 against 0.00049 nm) because resampling individual frames destroys precisely the correlation that matters; the block bootstrap with 25-frame blocks returns 0.00160 nm, about 3.2 times larger, in line with the autocorrelation correction. The block-length scan shows the usual behaviour: too short a block (5 frames = 10 ps, below \( \tau_{\text{int}} \)) underestimates the error at 0.00091 nm, and from 50 ps upwards the estimate plateaus between 0.0016 and 0.0019 nm with increasing Monte-Carlo noise as the number of distinct blocks falls. Choosing a block length is the price of the method, and the autocorrelation time is how you choose it.

Writing σ/√500 instead of σ/√n<sub>eff</sub> understates the uncertainty of a simulation average by a factor of about 4 for this run. The cleanest way to count samples correctly, though, is not to count frames at all, but to treat each **replica** as one measurement.



In [ ]:
per_run <- md_obs[, .(mean = mean(rmsd_ca_nm), sd = sd(rmsd_ca_nm), n = .N), by = .(condition, replica)]
A <- per_run[condition == "A", mean]; B <- per_run[condition == "B", mean]
frames_A <- md_obs[condition == "A", rmsd_ca_nm]
se_naive <- sd(frames_A) / sqrt(length(frames_A))
tau_A    <- mean(sapply(1:3, function(r) tau_int(md_obs[condition == "A" & replica == r, rmsd_ca_nm]) * dt))
se_tau   <- sd(frames_A) / sqrt(3 * 1000 / (2 * tau_A))
se_rep   <- sd(A) / sqrt(3)
bars <- data.table(method = factor(c("naive: 1500 frames", "autocorrelation-corrected", "replica-level (n = 3)"),
                                   levels = c("replica-level (n = 3)", "autocorrelation-corrected", "naive: 1500 frames")),
                   se = c(se_naive, se_tau, se_rep))
p <- ggplot(bars, aes(mean(A), method)) +
  geom_errorbarh(aes(xmin = mean(A) - 1.96 * se, xmax = mean(A) + 1.96 * se), height = 0.18,
                 colour = PAL["blue"]) +
  geom_point(size = 2, colour = PAL["blue"]) +
  geom_point(data = data.table(x = A, method = factor("replica-level (n = 3)", levels = levels(bars$method))),
             aes(x, method), shape = 4, size = 2.4, colour = PAL["orange"]) +
  labs(x = "mean C-alpha RMSD at 300 K (nm), with a nominal 95 % interval", y = NULL,
       title = "The same data, three error bars, one of them honest")
print(p)
print(per_run)
cat(sprintf("condition A replica means (nm): %s -> mean %.4f, SE %.4f\n",
            paste(sprintf("%.4f", A), collapse = ", "), mean(A), se_rep))
cat(sprintf("condition B replica means (nm): %s -> mean %.4f, SE %.4f\n",
            paste(sprintf("%.4f", B), collapse = ", "), mean(B), sd(B) / sqrt(3)))
cat(sprintf("95%% half-width: naive %.4f | tau-corrected %.4f | replica-level %.4f nm\n",
            1.96 * se_naive, 1.96 * se_tau, 1.96 * se_rep))
cat(sprintf("ratio replica-level / naive: %.1f x wider\n", se_rep / se_naive))



<p class="caption"><b>Figure 15.</b> The mean C-alpha RMSD of the three 300 K runs with three different error bars (orange crosses: the three replica means). The naive interval built from 1,500 frames is ±0.0031 nm; correcting for autocorrelation widens it to ±0.0180 nm; the replica-level interval, built from the three run means of 0.1098, 0.1595 and 0.2515 nm, is ±0.0813 nm — the only one that contains all three replicas. The replica-level standard error is 26.4 times the naive one. The runs disagree with each other far more than the frames within a run do, because each explores a different region of conformational space in 1 ns.</p>

The replica-level standard error is the honest one because it is the only one whose "repeat the experiment" matches what we would actually repeat. This is the statistical statement of a principle with a name — **pseudoreplication**<sup class="c"><a href="#r34">[34]</a></sup>, measuring the same experimental unit many times and counting the measurements as independent samples — and <a href="../U8_Hypothesis_Testing/index.html">Unit 8</a> turns it into its centrepiece experiment by running the same comparison at frame level and at replica level and getting p-values that differ by twenty-five orders of magnitude.

## 13. Putting it together: how the group reports an estimate

Every table and every figure in a manuscript from this laboratory should be reducible to the pattern below: an estimate, an interval, the n that produced it, and a statement of which source of variability the interval covers.



In [ ]:
ci_boot_med <- function(x, B = 4000, seed = 0) {
  set.seed(seed)
  bb <- boot(x, function(v, i) median(v[i]), R = B)
  as.vector(boot.ci(bb, type = "perc")$percent[4:5])
}
tt <- t.test(exact$pActivity)
m1 <- ci_boot_med(s1); m2 <- ci_boot_med(s2)
report <- data.table(
  quantity = c("mean pActivity, all exact compounds", "median pActivity, series 1",
               "median pActivity, series 2", "Spearman rho, cLogP vs pActivity",
               "mean C-alpha RMSD at 300 K (nm)"),
  estimate = c(mean(exact$pActivity), median(s1), median(s2), ctab$spearman[1], mean(A)),
  ci_low   = c(tt$conf.int[1], m1[1], m2[1], ctab$rho_lo[1], mean(A) - 1.96 * se_rep),
  ci_high  = c(tt$conf.int[2], m1[2], m2[2], ctab$rho_hi[1], mean(A) + 1.96 * se_rep),
  n        = c(nrow(exact), length(s1), length(s2), nrow(exact), 3L),
  covers   = c("sampling only (parametric t)", "sampling only (bootstrap)", "sampling only (bootstrap)",
               "sampling only (bootstrap)", "between-replica (n = 3 runs)"))
knitr::kable(report, digits = 3)



Read the last column as carefully as the first. None of these intervals covers the measurement error quantified in §11, and the last one covers between-replica variability but not force-field error. An interval is always an interval *with respect to a specified repetition*, and naming that repetition is part of reporting the number.

<div class="box bridge"><h4>Bridge</h4>
<p>This unit is the foundation of the statistics block of the R edition. <a href="../U8_Hypothesis_Testing/index.html">Unit 8</a> turns intervals into tests, effect sizes and multiplicity corrections, and reuses both the scaffold-series comparison and the replica MD data of this unit; <a href="../U9_Linear_Models/index.html">Unit 9</a> puts the same inference machinery inside a regression with <code>lm()</code> and <code>glmnet</code>; <a href="../U10_Bayesian/index.html">Unit 10</a> re-derives intervals as statements about parameters rather than about procedures. The Python counterpart of this unit, with the same experiments in NumPy and SciPy, is <a href="../../Python/U7_Estimation/index.html">here</a>, and the complete R edition is indexed at <a href="../index.html">the course root</a>. Backwards, <a href="../U5_EDA/index.html">Unit 5</a> produced the distributions we have now modelled and <a href="../U6_Data_Quality/index.html">Unit 6</a> produced the curated table whose noise floor §11 measured; forward, <a href="../U12_Validation/index.html">Unit 12</a> bootstraps model metrics exactly as §8 bootstraps a median, <a href="../U17_Uncertainty_AD/index.html">Unit 17</a> builds prediction intervals for single molecules, and <a href="../U20_MD_Analytics/index.html">Unit 20</a> develops the block averaging and convergence diagnostics sketched in §12. In the sister courses: the cheminformatics unit on <a href="../../../Cheminformatics/U7_Data_Curation/index.html">data curation</a> explains where replicate measurements come from and the unit on <a href="../../../Cheminformatics/U12_Classical_QSAR/index.html">classical QSAR</a> uses the pActivity scale throughout; the bioinformatics course unit on <a href="../../../BGM324/05_Molecular_Dynamics/index.html">molecular dynamics</a> explains how the trajectories analysed in §12 are produced; and the R edition of the AI course, indexed <a href="../../../BGM411/R/index.html">here</a>, uses the same resampling logic for learning curves.</p></div>

## 14. Summary

You should be able to explain, without notes:

- the difference between a population, a sample, an estimator and an estimate, and why only the estimator has a standard error;
- which of the five standard distributions a laboratory column plausibly follows, and how the index of dispersion tests a Poisson assumption (0.24 for hydrogen bonds, 2.88 for burst-like excursions);
- why potencies are averaged on the pActivity scale, and that the mean pIC50 is the logarithm of the geometric mean IC50;
- what the law of large numbers and the central limit theorem each promise, and that the CLT describes the mean and not the data;
- how to compute a parametric interval with `qt()` or `t.test()`, check its coverage by simulation, and why a nominal 95 % interval covered only 57 % of the time on the raw concentration scale;
- how to bootstrap any statistic with `boot()`, percentile versus BCa, and the three situations in which the bootstrap breaks;
- the breakdown points of the mean, trimmed mean and median, and when robustness is worth its cost;
- the difference between Pearson, Spearman and Kendall correlation, and why a coefficient without an interval is not evidence;
- how to estimate the noise floor of a bioactivity dataset (0.41 log units here) and what ceiling it places on any model's R² (0.87);
- why 500 molecular-dynamics frames were worth 28 independent samples, and why the replica-level error bar was 26 times the naive one.

## 15. Self-assessment

<details><summary>1. A colleague reports "mean IC50 = 4.2 µM" for a series whose median is 367 nM. What went wrong, and what should be reported instead?</summary>The arithmetic mean was computed on the concentration scale, where a handful of very weak compounds dominate the sum; in this dataset the weakest one per cent of the compounds contribute 25.4 % of the total. Report the median IC50, or equivalently the mean pActivity back-converted (the geometric mean, 311 nM here), together with an interval.</details>

<details><summary>2. Why is the standard deviation of the data not an error bar for the mean?</summary>Because they describe different things: the SD describes how spread out the individual observations are and does not shrink as more data are collected, while the standard error describes how precisely the mean is known and shrinks as σ/√n. Confusing them is the most common error-bar mistake in biology figures.</details>

<details><summary>3. Your 95 % confidence interval for a mean is [6.2, 6.8]. Is there a 95 % probability that the true mean lies in it?</summary>Not under the frequentist definition. The interval either contains the true value or it does not; the 95 % refers to the long-run performance of the procedure. That probability statement belongs to a Bayesian credible interval, which Unit 10 constructs.</details>

<details><summary>4. You bootstrap the median of 8 measurements and get a very narrow interval. Should you trust it?</summary>No. With eight points the bootstrap can only recombine eight observed values, so the replicate distribution is coarse and can be artificially tight — the series-2 histogram in Figure 8 already shows visible granularity with 59 values. Report the raw values, or use a parametric interval if the distributional assumption is defensible.</details>

<details><summary>5. The index of dispersion of a count column is 3.5. What does that tell you?</summary>That the counts are over-dispersed relative to Poisson, usually because the underlying rate varies between units (different cells, different genes, different frames) or because events arrive in bursts, as the cage-distance excursions do at 2.88. A negative binomial model, a quasi-Poisson correction or a resampling approach is appropriate; a Poisson-based interval would be far too narrow.</details>

<details><summary>6. Your model achieves a cross-validated RMSE of 0.25 log units on this dataset. What is your first reaction?</summary>Suspicion, not celebration. The pooled replicate standard deviation of the labels is 0.406 log units, so a model apparently more accurate than the measurements themselves almost certainly has leakage — duplicate structures across folds, a descriptor computed from the target, or a test set containing near-twins of training compounds. Check the split before anything else.</details>

<details><summary>7. Why is the Spearman correlation of −0.055 between cLogP and pActivity "significant" at p = 0.004?</summary>Because with 2,764 compounds the standard error of the coefficient is about 0.019, so even a trivially small association is a few standard errors from zero. The p-value answers "is it exactly zero?", which is almost never the question; the confidence interval, −0.091 to −0.017, answers "how big could it be?" and shows that the answer is "not big enough to matter".</details>

<details><summary>8. A 1 ns trajectory saved every 2 ps gives 500 frames. How many independent measurements of the RMSD is that?</summary>Roughly 1000 ps / (2 τ_int). With the estimator used here, τ_int for the six runs lies between about 18 and 80 ps, giving between 6 and 28 effectively independent samples per run. Any standard error computed as σ/√500 is too small by a factor of four to nine. Note that the value depends on how the autocorrelation sum is truncated, so quote the estimator along with the number.</details>

<details><summary>9. When is the percentile bootstrap interval not good enough?</summary>When the estimator is biased, or its sampling distribution is skewed, or its variance depends on the parameter value — for example a variance ratio, an enrichment factor or a small-sample correlation. BCa corrects for the bias (z0) and the skewness (a) and should be the default for anything other than a mean; the cost is the jackknife loop, which is negligible for samples of a few thousand. In R it is one extra word: <code>boot.ci(b, type = "bca")</code>.</details>

<details><summary>10. Two laboratories report IC50 values of 20 nM and 200 nM for the same compound. Is one of them wrong?</summary>Not necessarily. A one-log-unit disagreement is inside the tail of what public bioactivity data routinely show — this dataset contains sixty compounds with a range above one log unit, and the largest range is 3.30. Different assay formats, ATP concentrations and read-outs shift IC50 values systematically. Record both, flag the compound, and do not let it drive a model.</details>

<details><summary>11. Why does the mean of the pIC50 values equal the logarithm of the geometric mean of the IC50 values?</summary>Because the logarithm turns products into sums: mean(log x) = (1/n) Σ log x_i = log((Π x_i)^(1/n)). Averaging on the log scale is averaging the exponents, which is the geometric mean on the original scale. The worked example in §4 shows both routes giving 21.87 nM, and the <code>stopifnot</code> in that chunk enforces it.</details>

<details><summary>12. You must report "the RMSD at 340 K" from three replicas. What do you write?</summary>The mean of the three replica means with its standard error over n = 3 — here 0.1533 ± 0.0195 nm — stating explicitly that the interval reflects between-replica variability at 1 ns, together with the three individual run means (0.1211, 0.1884, 0.1504 nm) so the reader can see the spread. Never the frame-level standard error, and never a single replica.</details>

## Glossary

<dl class="gloss">
<dt>Autocorrelation time (τ<sub>int</sub>)</dt><dd>Integral of the autocorrelation function of a time series; two successive samples separated by less than τ<sub>int</sub> carry essentially the same information.</dd>
<dt>BCa interval</dt><dd>Bias-corrected and accelerated bootstrap interval; adjusts the percentiles for the bias and skewness of the bootstrap distribution. <code>boot.ci(type = "bca")</code>.</dd>
<dt>Block bootstrap</dt><dd>Resampling consecutive blocks of a time series instead of single observations, so that short-range dependence survives; <code>tsboot()</code> in the <code>boot</code> package.</dd>
<dt>Bootstrap</dt><dd>Resampling the observed data with replacement to approximate the sampling distribution of any statistic.</dd>
<dt>Breakdown point</dt><dd>The fraction of observations that must be corrupted before an estimator can be made arbitrarily wrong; 0 for the mean, 0.5 for the median.</dd>
<dt>Central limit theorem</dt><dd>The sampling distribution of a mean of independent draws with finite variance tends to a normal distribution as n grows.</dd>
<dt>Confidence interval</dt><dd>An interval produced by a procedure that, over repeated experiments, contains the true parameter a stated proportion of the time.</dd>
<dt>Effective sample size</dt><dd>The number of independent observations a correlated sample is worth, approximately n Δt / (2 τ<sub>int</sub>) for a time series.</dd>
<dt>Estimator / estimate</dt><dd>The rule for computing a quantity from data / the number that rule returns on a particular sample.</dd>
<dt>Geometric mean</dt><dd>The n-th root of the product of n positive values; equals the back-transformed mean of the logarithms.</dd>
<dt>Index of dispersion</dt><dd>Variance divided by mean; equals 1 for a Poisson variable, below 1 for constrained counts and above 1 for over-dispersed ones.</dd>
<dt>MAD</dt><dd>Median absolute deviation; multiplied by 1.4826 (R's <code>mad()</code> default) it estimates the standard deviation of normal data with a breakdown point of 0.5.</dd>
<dt>Noise floor</dt><dd>The reproducibility limit of the measurements themselves; the smallest error any model can achieve against those labels.</dd>
<dt>pActivity (pIC50)</dt><dd>−log<sub>10</sub> of a potency expressed in molar units; the scale on which potencies are averaged and modelled.</dd>
<dt>Pseudoreplication</dt><dd>Treating repeated measurements of one experimental unit as independent replicates, which inflates the apparent sample size.</dd>
<dt>QQ-plot</dt><dd>Plot of sample quantiles against the quantiles of a reference distribution; straight means "same shape". <code>qqnorm()</code> plus <code>qqline()</code> in R.</dd>
<dt>Replica</dt><dd>An independent repetition of a simulation with a different random seed; the experimental unit of a molecular-dynamics study.</dd>
<dt>Standard error</dt><dd>The standard deviation of an estimator's sampling distribution; σ/√n for a mean of independent draws.</dd>
</dl>

## References and further reading

<ol class="refs">
<li id="r1">Ihaka R, Gentleman R. R: a language for data analysis and graphics. <em>J Comput Graph Stat</em>. 1996;5(3):299-314. <a href="https://doi.org/10.1080/10618600.1996.10474713">doi:10.1080/10618600.1996.10474713</a></li>
<li id="r2">Wickham H. <em>ggplot2: Elegant Graphics for Data Analysis</em>. 2nd ed. Springer; 2016. <a href="https://doi.org/10.1007/978-3-319-24277-4">doi:10.1007/978-3-319-24277-4</a></li>
<li id="r3">Canty A, Ripley BD. boot: Bootstrap R (S-Plus) Functions. R package version 1.3; 2024. Companion software to Davison AC, Hinkley DV, <em>Bootstrap Methods and their Application</em>. <a href="https://doi.org/10.1017/CBO9780511802843">doi:10.1017/CBO9780511802843</a></li>
<li id="r4">Cheng Y, Prusoff WH. Relationship between the inhibition constant (KI) and the concentration of inhibitor which causes 50 per cent inhibition (I50) of an enzymatic reaction. <em>Biochem Pharmacol</em>. 1973;22(23):3099-3108. <a href="https://doi.org/10.1016/0006-2952(73)90196-2">doi:10.1016/0006-2952(73)90196-2</a></li>
<li id="r5">Zdrazil B, Felix E, Hunter F, et al. The ChEMBL Database in 2023: a drug discovery platform spanning multiple bioactivity data types and time periods. <em>Nucleic Acids Res</em>. 2024;52(D1):D1180-D1192. <a href="https://doi.org/10.1093/nar/gkad1004">doi:10.1093/nar/gkad1004</a></li>
<li id="r6">Student. The probable error of a mean. <em>Biometrika</em>. 1908;6(1):1-25. <a href="https://doi.org/10.1093/biomet/6.1.1">doi:10.1093/biomet/6.1.1</a></li>
<li id="r7">Limpert E, Stahel WA, Abbt M. Log-normal distributions across the sciences: keys and clues. <em>BioScience</em>. 2001;51(5):341-352. <a href="https://doi.org/10.1641/0006-3568(2001)051[0341:LNDATS]2.0.CO;2">doi:10.1641/0006-3568(2001)051[0341:LNDATS]2.0.CO;2</a></li>
<li id="r8">Shapiro SS, Wilk MB. An analysis of variance test for normality (complete samples). <em>Biometrika</em>. 1965;52(3-4):591-611. <a href="https://doi.org/10.1093/biomet/52.3-4.591">doi:10.1093/biomet/52.3-4.591</a></li>
<li id="r9">Ghasemi A, Zahediasl S. Normality tests for statistical analysis: a guide for non-statisticians. <em>Int J Endocrinol Metab</em>. 2012;10(2):486-489. <a href="https://doi.org/10.5812/ijem.3505">doi:10.5812/ijem.3505</a></li>
<li id="r10">Altman DG, Bland JM. Standard deviations and standard errors. <em>BMJ</em>. 2005;331(7521):903. <a href="https://doi.org/10.1136/bmj.331.7521.903">doi:10.1136/bmj.331.7521.903</a></li>
<li id="r11">Cumming G, Fidler F, Vaux DL. Error bars in experimental biology. <em>J Cell Biol</em>. 2007;177(1):7-11. <a href="https://doi.org/10.1083/jcb.200611141">doi:10.1083/jcb.200611141</a></li>
<li id="r12">Krzywinski M, Altman N. Error bars. <em>Nat Methods</em>. 2013;10:921-922. <a href="https://doi.org/10.1038/nmeth.2659">doi:10.1038/nmeth.2659</a></li>
<li id="r13">Krzywinski M, Altman N. Importance of being uncertain. <em>Nat Methods</em>. 2013;10:809-810. <a href="https://doi.org/10.1038/nmeth.2613">doi:10.1038/nmeth.2613</a></li>
<li id="r14">Cumming G. The new statistics: why and how. <em>Psychol Sci</em>. 2014;25(1):7-29. <a href="https://doi.org/10.1177/0956797613504966">doi:10.1177/0956797613504966</a></li>
<li id="r15">Efron B. Bootstrap methods: another look at the jackknife. <em>Ann Stat</em>. 1979;7(1):1-26. <a href="https://doi.org/10.1214/aos/1176344552">doi:10.1214/aos/1176344552</a></li>
<li id="r16">Efron B, Tibshirani R. Bootstrap methods for standard errors, confidence intervals, and other measures of statistical accuracy. <em>Stat Sci</em>. 1986;1(1):54-75. <a href="https://doi.org/10.1214/ss/1177013815">doi:10.1214/ss/1177013815</a></li>
<li id="r17">Davison AC, Hinkley DV. <em>Bootstrap Methods and their Application</em>. Cambridge University Press; 1997. <a href="https://doi.org/10.1017/CBO9780511802843">doi:10.1017/CBO9780511802843</a></li>
<li id="r18">Efron B. Better bootstrap confidence intervals. <em>J Am Stat Assoc</em>. 1987;82(397):171-185. <a href="https://doi.org/10.1080/01621459.1987.10478410">doi:10.1080/01621459.1987.10478410</a></li>
<li id="r19">DiCiccio TJ, Efron B. Bootstrap confidence intervals. <em>Stat Sci</em>. 1996;11(3):189-228. <a href="https://doi.org/10.1214/ss/1032280214">doi:10.1214/ss/1032280214</a></li>
<li id="r20">Huber PJ. Robust estimation of a location parameter. <em>Ann Math Stat</em>. 1964;35(1):73-101. <a href="https://doi.org/10.1214/aoms/1177703732">doi:10.1214/aoms/1177703732</a></li>
<li id="r21">Rousseeuw PJ, Croux C. Alternatives to the median absolute deviation. <em>J Am Stat Assoc</em>. 1993;88(424):1273-1283. <a href="https://doi.org/10.1080/01621459.1993.10476408">doi:10.1080/01621459.1993.10476408</a></li>
<li id="r22">Bonett DG, Wright TA. Sample size requirements for estimating Pearson, Kendall and Spearman correlations. <em>Psychometrika</em>. 2000;65:23-28. <a href="https://doi.org/10.1007/BF02294183">doi:10.1007/BF02294183</a></li>
<li id="r23">Bishara AJ, Hittner JB. Testing the significance of a correlation with nonnormal data: comparison of Pearson, Spearman, transformation, and resampling approaches. <em>Psychol Methods</em>. 2012;17(3):399-417. <a href="https://doi.org/10.1037/a0028087">doi:10.1037/a0028087</a></li>
<li id="r24">Anscombe FJ. Graphs in statistical analysis. <em>Am Stat</em>. 1973;27(1):17-21. <a href="https://doi.org/10.1080/00031305.1973.10478966">doi:10.1080/00031305.1973.10478966</a></li>
<li id="r25">Matejka J, Fitzmaurice G. Same stats, different graphs: generating datasets with varied appearance and identical statistics through simulated annealing. In: <em>Proc. CHI 2017</em>. ACM; 2017:1290-1294. <a href="https://doi.org/10.1145/3025453.3025912">doi:10.1145/3025453.3025912</a></li>
<li id="r26">Kalliokoski T, Kramer C, Vulpetti A, Gedeck P. Comparability of mixed IC50 data - a statistical analysis. <em>PLoS ONE</em>. 2013;8(4):e61007. <a href="https://doi.org/10.1371/journal.pone.0061007">doi:10.1371/journal.pone.0061007</a></li>
<li id="r27">Kramer C, Kalliokoski T, Gedeck P, Vulpetti A. The experimental uncertainty of heterogeneous public Ki data. <em>J Med Chem</em>. 2012;55(11):5165-5173. <a href="https://doi.org/10.1021/jm300131x">doi:10.1021/jm300131x</a></li>
<li id="r28">Neidigh JW, Fesinmeyer RM, Andersen NH. Designing a 20-residue protein. <em>Nat Struct Biol</em>. 2002;9:425-430. <a href="https://doi.org/10.1038/nsb798">doi:10.1038/nsb798</a></li>
<li id="r29">Eastman P, Swails J, Chodera JD, et al. OpenMM 7: rapid development of high performance algorithms for molecular dynamics. <em>PLoS Comput Biol</em>. 2017;13(7):e1005659. <a href="https://doi.org/10.1371/journal.pcbi.1005659">doi:10.1371/journal.pcbi.1005659</a></li>
<li id="r30">Maier JA, Martinez C, Kasavajhala K, Wickstrom L, Hauser KE, Simmerling C. ff14SB: improving the accuracy of protein side chain and backbone parameters from ff99SB. <em>J Chem Theory Comput</em>. 2015;11(8):3696-3713. <a href="https://doi.org/10.1021/acs.jctc.5b00255">doi:10.1021/acs.jctc.5b00255</a></li>
<li id="r31">Flyvbjerg H, Petersen HG. Error estimates on averages of correlated data. <em>J Chem Phys</em>. 1989;91(1):461-466. <a href="https://doi.org/10.1063/1.457480">doi:10.1063/1.457480</a></li>
<li id="r32">Grossfield A, Zuckerman DM. Quantifying uncertainty and sampling quality in biomolecular simulations. <em>Annu Rep Comput Chem</em>. 2009;5:23-48. <a href="https://doi.org/10.1016/S1574-1400(09)00502-7">doi:10.1016/S1574-1400(09)00502-7</a></li>
<li id="r33">Chodera JD. A simple method for automated equilibration detection in molecular simulations. <em>J Chem Theory Comput</em>. 2016;12(4):1799-1805. <a href="https://doi.org/10.1021/acs.jctc.5b00784">doi:10.1021/acs.jctc.5b00784</a></li>
<li id="r34">Hurlbert SH. Pseudoreplication and the design of ecological field experiments. <em>Ecol Monogr</em>. 1984;54(2):187-211. <a href="https://doi.org/10.2307/1942661">doi:10.2307/1942661</a></li>
</ol>

<h4>Textbooks</h4>
<ul>
<li>Wasserman L. <em>All of Statistics: A Concise Course in Statistical Inference</em>. Springer; 2004. — Chapters 1–9 cover everything in this unit at a mathematical level one step above it.</li>
<li>Efron B, Tibshirani RJ. <em>An Introduction to the Bootstrap</em>. Chapman &amp; Hall; 1993. — The book-length treatment of §8, including the BCa derivation.</li>
<li>Wickham H, Çetinkaya-Rundel M, Grolemund G. <em>R for Data Science</em>. 2nd ed. O'Reilly; 2023 (free online). — The R idioms used throughout this edition.</li>
<li>James G, Witten D, Hastie T, Tibshirani R. <em>An Introduction to Statistical Learning with Applications in R</em>. 2nd ed. Springer; 2021 (free PDF). — Chapter 5 for resampling as it is used in modelling.</li>
<li>McElreath R. <em>Statistical Rethinking</em>. 2nd ed. CRC Press; 2020. — Chapter 4 motivates the normal distribution from the same simulation-first angle; the Bayesian counterpart is Unit 10.</li>
</ul>

<h4>Online resources</h4>
<ul>
<li>The <code>boot</code> package reference manual and the <code>tsboot</code> help page: <a href="https://cran.r-project.org/package=boot">cran.r-project.org/package=boot</a>.</li>
<li>Nature Methods "Points of Significance" column (2013–2016), a two-page-per-topic companion to this unit: <a href="https://www.nature.com/collections/qghhqm">nature.com/collections/qghhqm</a>.</li>
<li>Practical Cheminformatics (P. Walters), posts on the reproducibility of public bioactivity data: <a href="https://patwalters.github.io/practicalcheminformatics/">patwalters.github.io/practicalcheminformatics</a>.</li>
<li>LiveCoMS, "Best Practices for Quantification of Uncertainty and Sampling Quality in Molecular Simulations": <a href="https://livecomsjournal.org/">livecomsjournal.org</a>.</li>
</ul>
